In [0]:
# =========================================================
# GOLD LAYER — WITH ERROR HANDLING + LOGGING
# =========================================================

import builtins
import uuid
import traceback
from functools import reduce
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# ── Inline PipelineLogger (same as silver) ────────────────

ERROR_LOG_SCHEMA = StructType([
    StructField("log_id",          StringType(),    False),
    StructField("pipeline_run_id", StringType(),    False),
    StructField("layer",           StringType(),    False),
    StructField("table_name",      StringType(),    False),
    StructField("step",            StringType(),    False),
    StructField("error_type",      StringType(),    True),
    StructField("error_message",   StringType(),    True),
    StructField("stack_trace",     StringType(),    True),
    StructField("rows_expected",   LongType(),      True),
    StructField("rows_written",    LongType(),      True),
    StructField("is_critical",     BooleanType(),   False),
    StructField("logged_at",       TimestampType(), False),
])

RUN_LOG_SCHEMA = StructType([
    StructField("run_id",        StringType(),    False),
    StructField("layer",         StringType(),    False),
    StructField("started_at",    TimestampType(), False),
    StructField("ended_at",      TimestampType(), True),
    StructField("status",        StringType(),    False),
    StructField("tables_ok",     IntegerType(),   True),
    StructField("tables_failed", IntegerType(),   True),
    StructField("notes",         StringType(),    True),
])

class PipelineLogger:
    def __init__(self, spark, layer):
        self.spark         = spark
        self.layer         = layer.upper()
        self.run_id        = builtins.str(uuid.uuid4())
        self.tables_ok     = 0
        self.tables_failed = 0
        self.started_at    = datetime.now()
        spark.sql("CREATE SCHEMA IF NOT EXISTS traffic_catalog.logs")
        spark.sql("""
            CREATE TABLE IF NOT EXISTS traffic_catalog.logs.error_log (
                log_id STRING, pipeline_run_id STRING, layer STRING,
                table_name STRING, step STRING, error_type STRING,
                error_message STRING, stack_trace STRING,
                rows_expected LONG, rows_written LONG,
                is_critical BOOLEAN, logged_at TIMESTAMP
            ) USING DELTA
        """)
        spark.sql("""
            CREATE TABLE IF NOT EXISTS traffic_catalog.logs.pipeline_run_log (
                run_id STRING, layer STRING, started_at TIMESTAMP,
                ended_at TIMESTAMP, status STRING,
                tables_ok INT, tables_failed INT, notes STRING
            ) USING DELTA
        """)
        self._write_run("RUNNING", "Pipeline started")
        print(f"[{self.layer}] run_id: {self.run_id} | started: {self.started_at}")

    def log_error(self, table, step, error, is_critical=False,
                  rows_expected=None, rows_written=None):
        row = {
            "log_id":          builtins.str(uuid.uuid4()),
            "pipeline_run_id": self.run_id,
            "layer":           self.layer,
            "table_name":      table,
            "step":            step.upper(),
            "error_type":      type(error).__name__,
            "error_message":   builtins.str(error)[:2000],
            "stack_trace":     traceback.format_exc()[:4000],
            "rows_expected":   rows_expected,
            "rows_written":    rows_written,
            "is_critical":     is_critical,
            "logged_at":       datetime.now(),
        }
        self.spark.createDataFrame([row], ERROR_LOG_SCHEMA) \
            .write.format("delta").mode("append") \
            .saveAsTable("traffic_catalog.logs.error_log")
        lvl = "CRITICAL" if is_critical else "WARNING"
        print(f"  [{lvl}] {table}.{step} — {type(error).__name__}: {builtins.str(error)[:150]}")

    def log_success(self, table, rows):
        self.tables_ok += 1
        print(f"  [OK] {table} — {rows:,} rows")

    def log_failure(self, table):
        self.tables_failed += 1

    def finish(self):
        status = "FAILED" if self.tables_failed > 0 else "SUCCESS"
        self._write_run(status, f"{self.tables_ok} ok / {self.tables_failed} failed")
        print(f"\n[{self.layer}] {status} | ok: {self.tables_ok} | failed: {self.tables_failed}")
        return status

    def _write_run(self, status, notes=None):
        row = {
            "run_id": self.run_id, "layer": self.layer,
            "started_at": self.started_at,
            "ended_at": datetime.now() if status != "RUNNING" else None,
            "status": status, "tables_ok": self.tables_ok,
            "tables_failed": self.tables_failed, "notes": notes,
        }
        self.spark.createDataFrame([row], RUN_LOG_SCHEMA) \
            .write.format("delta").mode("append") \
            .saveAsTable("traffic_catalog.logs.pipeline_run_log")

def run_step(logger, table, step, fn, is_critical=True):
    try:
        return fn()
    except Exception as e:
        logger.log_error(table, step, e, is_critical=is_critical)
        logger.log_failure(table)
        if is_critical:
            logger.finish()
            raise RuntimeError(
                f"Critical failure: {table}.{step} — run_id={logger.run_id}"
            ) from e
        return None

# =========================================================
# START PIPELINE
# =========================================================

logger = PipelineLogger(spark, layer="GOLD")

spark.sql("CREATE SCHEMA IF NOT EXISTS traffic_catalog.gold")

# ── Helper ────────────────────────────────────────────────
def write_gold(df, table_name, logger, partition_cols=None):
    def _write():
        writer = df.write.format("delta").mode("overwrite") \
                   .option("overwriteSchema", "true")
        if partition_cols:
            writer = writer.partitionBy(*partition_cols)
        writer.saveAsTable(f"traffic_catalog.gold.{table_name}")
        rows = spark.table(f"traffic_catalog.gold.{table_name}").count()
        logger.log_success(table_name, rows)
        return rows
    return run_step(logger, table_name, "WRITE", _write, is_critical=True)

# =========================================================
# DIMENSION TABLES
# =========================================================

print("\n── DIMENSION TABLES ──")

# dim_date
def _build_dim_date():
    sources = [spark.table(f"traffic_catalog.silver.{t}").select("event_date")
               for t in ["incident_clean","sensor_clean","speed_clean",
                         "vehicle_clean","signal_clean"]]
    all_dates = reduce(lambda a, b: a.union(b), sources).distinct()
    return all_dates.select(
        F.date_format(F.col("event_date"), "yyyyMMdd").cast(IntegerType()).alias("date_key"),
        F.col("event_date").alias("full_date"),
        F.year(F.col("event_date")).alias("year"),
        F.month(F.col("event_date")).alias("month"),
        F.date_format(F.col("event_date"), "MMMM").alias("month_name"),
        F.dayofmonth(F.col("event_date")).alias("day"),
        F.dayofweek(F.col("event_date")).alias("day_of_week"),
        F.date_format(F.col("event_date"), "EEEE").alias("weekday_name"),
        F.weekofyear(F.col("event_date")).alias("week_of_year"),
        F.quarter(F.col("event_date")).alias("quarter"),
        F.concat(F.lit("Q"), F.quarter(F.col("event_date"))).alias("quarter_label"),
        F.when(F.dayofweek(F.col("event_date")).isin(1, 7), F.lit(True))
         .otherwise(F.lit(False)).alias("is_weekend")
    ).orderBy("date_key")

dim_date = run_step(logger, "dim_date", "BUILD", _build_dim_date, is_critical=True)
if dim_date is not None:
    write_gold(dim_date, "dim_date", logger)

# dim_time
def _build_dim_time():
    sources = [spark.table(f"traffic_catalog.silver.{t}").select("event_hour","event_minute")
               for t in ["incident_clean","sensor_clean","speed_clean",
                         "vehicle_clean","signal_clean"]]
    all_times = reduce(lambda a, b: a.union(b), sources).distinct()
    return all_times.select(
        (F.col("event_hour") * 100 + F.col("event_minute")).cast(IntegerType()).alias("time_key"),
        F.col("event_hour").alias("hour"),
        F.col("event_minute").alias("minute"),
        F.concat(F.lpad(F.col("event_hour").cast(StringType()), 2, "0"),
                 F.lit(":"),
                 F.lpad(F.col("event_minute").cast(StringType()), 2, "0")).alias("time_label"),
        F.when((F.col("event_hour") >= 6)  & (F.col("event_hour") < 10), F.lit("MORNING_PEAK"))
         .when((F.col("event_hour") >= 10) & (F.col("event_hour") < 16), F.lit("MIDDAY"))
         .when((F.col("event_hour") >= 16) & (F.col("event_hour") < 20), F.lit("EVENING_PEAK"))
         .when((F.col("event_hour") >= 20) & (F.col("event_hour") < 24), F.lit("NIGHT"))
         .otherwise(F.lit("LATE_NIGHT")).alias("time_of_day"),
        F.when(((F.col("event_hour") >= 6)  & (F.col("event_hour") < 10)) |
               ((F.col("event_hour") >= 16) & (F.col("event_hour") < 20)),
               F.lit(True)).otherwise(F.lit(False)).alias("is_peak_hour"),
        F.when((F.col("event_hour") >= 6)  & (F.col("event_hour") < 14), F.lit("MORNING_SHIFT"))
         .when((F.col("event_hour") >= 14) & (F.col("event_hour") < 22), F.lit("AFTERNOON_SHIFT"))
         .otherwise(F.lit("NIGHT_SHIFT")).alias("shift")
    ).orderBy("time_key")

dim_time = run_step(logger, "dim_time", "BUILD", _build_dim_time, is_critical=True)
if dim_time is not None:
    write_gold(dim_time, "dim_time", logger)

# dim_sensor
def _build_dim_sensor():
    w = Window.partitionBy("sensor_id").orderBy(F.col("event_timestamp").desc())
    return spark.table("traffic_catalog.silver.sensor_clean") \
        .withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn") \
        .select(
            F.col("sensor_id"),
            F.col("location_id"),
            F.col("lane_id"),
            F.when(F.col("sensor_id").startswith("S"), F.lit("SPEED_SENSOR"))
             .when(F.col("sensor_id").startswith("T"), F.lit("TRAFFIC_SENSOR"))
             .otherwise(F.lit("STANDARD")).alias("sensor_type"),
            F.lit(True).alias("is_active")
        ).orderBy("sensor_id")

dim_sensor = run_step(logger, "dim_sensor", "BUILD", _build_dim_sensor, is_critical=True)
if dim_sensor is not None:
    write_gold(dim_sensor, "dim_sensor", logger)

# dim_vehicle
def _build_dim_vehicle():
    return spark.table("traffic_catalog.silver.vehicle_clean") \
        .select("vehicle_type").distinct() \
        .withColumn("is_heavy_vehicle", F.col("vehicle_type").isin("TRUCK","BUS")) \
        .withColumn("vehicle_category",
            F.when(F.col("vehicle_type").isin("TRUCK","BUS"), F.lit("HEAVY"))
             .when(F.col("vehicle_type") == "CAR",  F.lit("LIGHT"))
             .when(F.col("vehicle_type") == "BIKE", F.lit("TWO_WHEELER"))
             .otherwise(F.lit("OTHER"))) \
        .withColumn("vehicle_description",
            F.when(F.col("vehicle_type") == "CAR",   F.lit("Passenger car"))
             .when(F.col("vehicle_type") == "TRUCK", F.lit("Heavy goods vehicle"))
             .when(F.col("vehicle_type") == "BUS",   F.lit("Public transport bus"))
             .when(F.col("vehicle_type") == "BIKE",  F.lit("Motorcycle or bicycle"))
             .otherwise(F.lit("Unknown"))) \
        .orderBy("vehicle_type")

dim_vehicle = run_step(logger, "dim_vehicle", "BUILD", _build_dim_vehicle, is_critical=True)
if dim_vehicle is not None:
    write_gold(dim_vehicle, "dim_vehicle", logger)

# dim_incident
def _build_dim_incident():
    return spark.table("traffic_catalog.silver.incident_clean") \
        .select("incident_type","incident_severity").distinct() \
        .withColumn("severity_rank",
            F.when(F.col("incident_severity") == "LOW",    F.lit(1))
             .when(F.col("incident_severity") == "MEDIUM", F.lit(2))
             .when(F.col("incident_severity") == "HIGH",   F.lit(3))
             .otherwise(F.lit(0))) \
        .withColumn("requires_immediate_response",
            F.col("incident_severity") == "HIGH") \
        .withColumn("incident_key",
            F.concat(F.col("incident_type"), F.lit("_"), F.col("incident_severity"))) \
        .orderBy("incident_type","severity_rank")

dim_incident = run_step(logger, "dim_incident", "BUILD", _build_dim_incident, is_critical=True)
if dim_incident is not None:
    write_gold(dim_incident, "dim_incident", logger)

# dim_signal
def _build_dim_signal():
    w = Window.partitionBy("signal_id").orderBy(F.col("event_timestamp").desc())
    return spark.table("traffic_catalog.silver.signal_clean") \
        .withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn") \
        .select(
            F.col("signal_id"),
            F.col("sensor_id"),
            F.when(F.col("signal_id").startswith("SIG"), F.lit("TRAFFIC_LIGHT"))
             .when(F.col("signal_id").startswith("PED"), F.lit("PEDESTRIAN"))
             .otherwise(F.lit("STANDARD")).alias("signal_type"),
            F.lit(True).alias("is_active")
        ).orderBy("signal_id")

dim_signal = run_step(logger, "dim_signal", "BUILD", _build_dim_signal, is_critical=True)
if dim_signal is not None:
    write_gold(dim_signal, "dim_signal", logger)

# =========================================================
# DIMENSION VALIDATION
# =========================================================

print("\n── DIMENSION VALIDATION ──")

dim_tables = {
    "dim_date":     "date_key",
    "dim_time":     "time_key",
    "dim_sensor":   "sensor_id",
    "dim_vehicle":  "vehicle_type",
    "dim_incident": "incident_key",
    "dim_signal":   "signal_id"
}

all_dims_ok = True
for table, pk in dim_tables.items():
    def _validate_dim(t=table, p=pk):
        df    = spark.table(f"traffic_catalog.gold.{t}")
        rows  = df.count()
        nulls = df.filter(F.col(p).isNull()).count()
        dups  = rows - df.select(p).distinct().count()
        if nulls > 0 or dups > 0:
            raise ValueError(f"{t} — PK nulls: {nulls}, PK dups: {dups}")
        print(f"  [OK] {t} — {rows:,} rows | PK nulls: 0 | PK dups: 0")
        return rows

    result = run_step(logger, table, "DIM_VALIDATE", _validate_dim, is_critical=True)
    if result is None:
        all_dims_ok = False

if not all_dims_ok:
    logger.finish()
    raise RuntimeError("Dimension validation failed — cannot build fact table safely")

# =========================================================
# FACT TABLE
# =========================================================

print("\n── FACT TABLE ──")

join_keys = ["sensor_id", "event_date", "event_hour"]

def deduplicate_for_join(df, keys):
    w = Window.partitionBy(*keys).orderBy(F.col("event_timestamp").desc())
    return df.withColumn("_rn", F.row_number().over(w)) \
             .filter(F.col("_rn") == 1).drop("_rn")

def _build_fact():
    sensor_df   = deduplicate_for_join(
        spark.table("traffic_catalog.silver.sensor_clean"),   join_keys).alias("sen")
    incident_df = deduplicate_for_join(
        spark.table("traffic_catalog.silver.incident_clean"), join_keys).alias("inc")
    speed_df    = deduplicate_for_join(
        spark.table("traffic_catalog.silver.speed_clean"),    join_keys).alias("spd")
    signal_df   = deduplicate_for_join(
        spark.table("traffic_catalog.silver.signal_clean"),   join_keys).alias("sig")
    vehicle_df  = deduplicate_for_join(
        spark.table("traffic_catalog.silver.vehicle_clean"),  join_keys).alias("veh")

    return (sensor_df
        .join(incident_df, on=join_keys, how="left")
        .join(speed_df,    on=join_keys, how="left")
        .join(signal_df,   on=join_keys, how="left")
        .join(vehicle_df,  on=join_keys, how="left")
        .select(
            F.col("sen.event_id").alias("event_id"),
            F.col("sen.sensor_id").alias("sensor_id"),
            F.date_format(F.col("sen.event_date"), "yyyyMMdd")
             .cast(IntegerType()).alias("date_key"),
            (F.col("sen.event_hour") * 100).cast(IntegerType()).alias("time_key"),
            F.col("veh.vehicle_type").alias("vehicle_type"),
            F.concat(F.coalesce(F.col("inc.incident_type"),     F.lit("UNKNOWN")),
                     F.lit("_"),
                     F.coalesce(F.col("inc.incident_severity"), F.lit("UNKNOWN"))
            ).alias("incident_key"),
            F.col("sig.signal_id").alias("signal_id"),
            F.col("sen.event_timestamp").alias("event_timestamp"),
            F.col("sen.event_date").alias("event_date"),
            F.col("sen.event_year").alias("event_year"),
            F.col("sen.event_month").alias("event_month"),
            F.col("sen.vehicle_count").alias("vehicle_count"),
            F.col("sen.avg_speed").alias("avg_speed"),
            F.col("sen.traffic_density").alias("traffic_density"),
            F.col("sen.occupancy_rate").alias("occupancy_rate"),
            F.col("sen.congestion_score").alias("congestion_score"),
            F.col("inc.incident_id").alias("incident_id"),
            F.col("inc.incident_duration").alias("incident_duration"),
            F.col("inc.vehicles_affected").alias("vehicles_affected"),
            F.col("inc.response_time").alias("response_time"),
            F.col("inc.lane_blocked_flag").alias("lane_blocked_flag"),
            F.col("inc.is_severe").alias("is_severe"),
            F.col("spd.speed_limit").alias("speed_limit"),
            F.col("spd.min_speed").alias("min_speed"),
            F.col("spd.avg_speed").alias("sensor_avg_speed"),
            F.col("spd.max_speed").alias("max_speed"),
            F.col("spd.speed_variance").alias("speed_variance"),
            F.col("spd.speed_violation_count").alias("speed_violation_count"),
            F.col("spd.speed_index").alias("speed_index"),
            F.col("spd.speed_excess").alias("speed_excess"),
            F.col("spd.speed_violation_flag").alias("speed_violation_flag"),
            F.col("sig.signal_cycle_time").alias("signal_cycle_time"),
            F.col("sig.green_time").alias("green_time"),
            F.col("sig.red_time").alias("red_time"),
            F.col("sig.signal_wait_time").alias("signal_wait_time"),
            F.col("sig.signal_efficiency").alias("signal_efficiency"),
            F.col("sig.green_ratio").alias("green_ratio"),
            F.col("sig.red_ratio").alias("red_ratio"),
            F.col("sig.queue_length").alias("queue_length"),
            F.col("veh.vehicle_count").alias("vehicle_type_count"),
            F.col("veh.vehicle_flow_rate").alias("vehicle_flow_rate"),
            F.col("veh.vehicle_density").alias("vehicle_density"),
            F.col("veh.vehicle_delay").alias("vehicle_delay"),
            F.col("veh.lane_utilization").alias("lane_utilization"),
            F.col("sen.is_outlier").alias("is_outlier"),
            F.col("sen.dq_score").alias("dq_score"),
            F.when(F.col("inc.incident_id").isNotNull(), F.lit(True))
             .otherwise(F.lit(False)).alias("has_incident"),
            F.coalesce(F.col("spd.speed_violation_flag"), F.lit(False))
             .alias("has_speed_violation"),
            F.when(F.col("sig.signal_id").isNotNull(), F.lit(True))
             .otherwise(F.lit(False)).alias("has_signal"),
            F.col("sen.is_peak_hour").alias("is_peak_hour")
        )
    )

fact_df = run_step(logger, "fact_traffic_events", "BUILD", _build_fact, is_critical=True)

if fact_df is not None:
    write_gold(fact_df, "fact_traffic_events", logger,
               partition_cols=["event_year", "event_month"])

# =========================================================
# GOLD DQ VALIDATION
# =========================================================

print("\n── GOLD DQ VALIDATION ──")

def _validate_fact():
    fact = spark.table("traffic_catalog.gold.fact_traffic_events")
    total           = fact.count()
    null_sensors    = fact.filter(F.col("sensor_id").isNull()).count()
    null_date_keys  = fact.filter(F.col("date_key").isNull()).count()
    null_time_keys  = fact.filter(F.col("time_key").isNull()).count()
    negative_counts = fact.filter(F.col("vehicle_count") < 0).count()
    negative_speeds = fact.filter(F.col("avg_speed") < 0).count()

    # Referential integrity — all sensor_ids exist in dim_sensor
    dim_sensors   = spark.table("traffic_catalog.gold.dim_sensor").select("sensor_id")
    orphan_sensors = fact.select("sensor_id").distinct().subtract(dim_sensors).count()

    # All vehicle_types exist in dim_vehicle
    dim_vehicles   = spark.table("traffic_catalog.gold.dim_vehicle").select("vehicle_type")
    orphan_vehicles = (fact.select("vehicle_type")
                          .filter(F.col("vehicle_type").isNotNull())
                          .distinct().subtract(dim_vehicles).count())

    issues = []
    if null_sensors    > 0: issues.append(f"null sensor_ids: {null_sensors}")
    if null_date_keys  > 0: issues.append(f"null date_keys: {null_date_keys}")
    if null_time_keys  > 0: issues.append(f"null time_keys: {null_time_keys}")
    if negative_counts > 0: issues.append(f"negative vehicle_counts: {negative_counts}")
    if negative_speeds > 0: issues.append(f"negative avg_speeds: {negative_speeds}")
    if orphan_sensors  > 0: issues.append(f"orphan sensor_ids: {orphan_sensors}")
    if orphan_vehicles > 0: issues.append(f"orphan vehicle_types: {orphan_vehicles}")

    if issues:
        raise ValueError(f"Fact DQ failed: {'; '.join(issues)}")

    has_incident  = fact.filter(F.col("has_incident")        == True).count()
    has_violation = fact.filter(F.col("has_speed_violation") == True).count()
    has_signal    = fact.filter(F.col("has_signal")          == True).count()

    print(f"  [OK] fact_traffic_events — {total:,} rows")
    print(f"  Null checks     : sensor_ids={null_sensors} | date_keys={null_date_keys} | time_keys={null_time_keys}")
    print(f"  Range checks    : negative_counts={negative_counts} | negative_speeds={negative_speeds}")
    print(f"  Ref. integrity  : orphan_sensors={orphan_sensors} | orphan_vehicles={orphan_vehicles}")
    print(f"  Match rates     : incident={builtins.round(has_incident/total*100,1)}% | "
          f"violation={builtins.round(has_violation/total*100,1)}% | "
          f"signal={builtins.round(has_signal/total*100,1)}%")
    return total

run_step(logger, "fact_traffic_events", "GOLD_DQ_VALIDATE",
         _validate_fact, is_critical=False)

# =========================================================
# FINISH
# =========================================================

final_status = logger.finish()

print("\n==============================")
print("GOLD PIPELINE COMPLETE")
print("==============================")
print(f"Status  : {final_status}")
print(f"Run ID  : {logger.run_id}")
print(f"\nQuery error log:")
print(f"  SELECT * FROM traffic_catalog.logs.error_log "
      f"WHERE pipeline_run_id = '{logger.run_id}'")
print(f"\nQuery run history:")
print(f"  SELECT * FROM traffic_catalog.logs.pipeline_run_log ORDER BY started_at DESC")